In [1]:
import time
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import norm

from model import TransferModel
from config import SlimConfig
from data_processing import DataProcessor
from proxy_score import proxy_leaderboard   # your copy with the settings loop

warnings.filterwarnings("ignore", category=FutureWarning)

# ---- settings -------------------------------------------------------------
K, LAM, ALPHA = 20, 100, 2.0  # model config you'd submit (see validate_loco.ipynb)
CONTEXT = "A"                  # which context's controls to test against
N_TARGETS = 30                 # sampled challenge targets (more = slower, steadier numbers)
SEED = 0
FDR = 0.05                     # BH threshold for "significant"

In [2]:
DATASETS = {
    "K562": "../../data/replogle_nadig/K562_gwps_raw_bulk_01.h5ad",
    "RPE1": "../../data/replogle_nadig/rpe1_raw_bulk_01.h5ad",
    "HepG2": "../../data/replogle_nadig/nadig_hepg2_pseudobulk.h5ad",
    "Jurkat": "../../data/replogle_nadig/nadig_jurkat_pseudobulk.h5ad",
}

In [3]:
gene_names = pd.read_csv("../../data/controls/gene_names.csv")["gene_name"].tolist()
dp = DataProcessor(gene_names)
kd = 0.9
ctrl = ad.read_h5ad(f"../../data/controls/context_{CONTEXT}.h5ad")

In [4]:
HELD = "HepG2"                     # screen used as the proxy truth (RPE1, HepG2 or Jurkat)
N_PROXY = 40                       # panel size

# LOCO fit: model trained without the held-out line
loco_train = [n for n in DATASETS if n != HELD]
loco = TransferModel(gene_names, '../../data/embeddings/gene_string_embeddings.v0.3.h5', SlimConfig())
loco.config.K, loco.config.lambda_reg = K, LAM
loco.fit(*dp.build_Y(loco_train, genes="union"))

held = dp.load_dataset(HELD)
truth = pd.DataFrame((held.X - held.var["ctrl_mean"].to_numpy()) / np.log(2),   # measured log2FC
                     index=held.obs_names, columns=[dp.gene_index[g] for g in held.var_names])
loco.set_context(held.var["ctrl_mean"], dp.source_ctrls(loco_train), tau=0.05, mix=0.3,
                 panel=list(truth.index))
print(HELD, dict(zip(loco.source_names, loco.weights.round(2))))

# targets the training lines also perturbed (like the VCC panel), and that are genes in the panel
trained_on = set().union(*(dp.load_dataset(n).obs_names for n in loco_train))
candidates = [t for t in truth.index if t in trained_on and t in dp.gene_index]
proxy_targets = list(np.random.default_rng(SEED).choice(candidates, N_PROXY, replace=False))

  Fitted on 13823 perts, STRING coverage: 13315/13823
  direct transfer from ('K562',): 9675 targets have a measurement
HepG2 {'K562': np.float64(0.37), 'RPE1': np.float64(0.37), 'Jurkat': np.float64(0.26)}


In [5]:
settings = {"old: gain 2.5 all":   {"gain": 2.5},
            "N 200, top 2.5":      {"gain": 1.0, "n_call": 200, "gain_top": 2.5},
            "N 500, top 2.5":      {"gain": 1.0, "n_call": 500, "gain_top": 2.5},
            "N 200, top 4":        {"gain": 1.0, "n_call": 200, "gain_top": 4.0}}
scaled, raw = proxy_leaderboard(loco, ctrl, truth, proxy_targets, settings, dp.gene_index,
                                kd=kd, alpha=ALPHA, seed=SEED)
display(raw.round(3))
scaled.round(3)

scorer: 9929 DE-tested genes, 9200 comparison / 9200 pool cells (4s)
reference + replicate: 53s, median n_real 328
old: gain 2.5 all: 25s
N 200, top 2.5: 26s
N 500, top 2.5: 25s
N 200, top 4: 25s


,pds,mse,fid,reach,jac,nmae,n_sig
baseline,0.500,0.872,0.534,0.013,0.043,0.914,9654.0
replicate,1.000,0.131,0.900,0.999,0.272,0.397,125.0
old: gain 2.5 all,0.550,2.395,0.613,0.256,0.142,0.988,1118.0
"N 200, top 2.5",0.553,1.506,0.470,0.213,0.119,0.993,293.0
"N 500, top 2.5",0.555,1.789,0.543,0.226,0.129,1.013,484.0
"N 200, top 4",0.557,2.708,0.490,0.203,0.114,1.172,361.5


,pds,mse,fid,reach,jac,nmae,overall,n_sig
baseline,0.000,-0.0,0.000,0.000,0.000,-0.000,0.000,9654.0
replicate,1.000,1.0,1.000,1.000,1.000,1.000,1.000,125.0
old: gain 2.5 all,0.100,0.0,0.214,0.246,0.434,-0.143,0.142,1118.0
"N 200, top 2.5",0.106,0.0,-0.175,0.203,0.331,-0.152,0.052,293.0
"N 500, top 2.5",0.110,0.0,0.025,0.215,0.375,-0.191,0.089,484.0
"N 200, top 4",0.114,0.0,-0.121,0.193,0.312,-0.499,-0.000,361.5
proxy reference,NaN,NaN,NaN,NaN,NaN,NaN,NaN,327.5
